# Eole によるニューラル機械翻訳演習
Eole というツールを用いて英日機械翻訳の演習を行います。Eole は、書籍で紹介している OpenNMT-py の後継のツールです。OpenNMT-py は2024年の v3.5.1 で開発が止まっているため、この演習では Eole を使います。コマンド名や設定ファイルの書き方は少し違いますが、できることはほぼ同じです。
Google Colaboratory は GPU を用いた実験ができるので、GPU を用いてニューラル機械翻訳の実験をしましょう。（「ランタイム」→「ランタイムのタイプを変更」から「ハードウェア アクセラレータ」を GPU（T4 など）にしておいてください。以下、Python 3 で説明します。）
データは田中コーパスという自由に使える対訳コーパスを使用します。翻訳結果を実際に見て評価する、ということも演習の目的です。

# Eole のインストール

## pip によるインストール
Eole を使って実験を行います。開発が速く、数週間で内容が変わってしまうことがあるので、ここでは2026年10月時点の最新版である v0.6.4 というバージョンを使います。pip というのは Python のソフトウェアをインストールするときに使うツールの名前で、依存関係も含めて自動でセットアップしてくれます。

ただし、2026年10月時点の Google Colab の Python は 3.13 ですが、Eole が使うライブラリ（pyonmttok や fasttext）は Python 3.12 までしか対応していないため、そのままでは `pip install eole` が失敗します。そこで、[uv](https://docs.astral.sh/uv/) という高速なパッケージ管理ツールを使って Python 3.12 の仮想環境を /content/eole-env に作り、そこに Eole と PyTorch をインストールします。1〜2分かかります。最後に `True` と表示されれば GPU が使えます。

In [ ]:
!pip install -q uv
!uv venv -q --clear -p 3.12 /content/eole-env
!uv pip install -q -p /content/eole-env eole==0.6.4
!/content/eole-env/bin/python -c "import sys, torch; print(sys.version, torch.__version__, torch.cuda.is_available())"

以下では `!eole ...` のようにコマンドを実行するので、作った仮想環境のコマンドが使えるように PATH を設定しておきます。ランタイムを再起動した場合は、このセルをもう一度実行してください。

In [ ]:
import os
os.environ["PATH"] = "/content/eole-env/bin:" + os.environ["PATH"]
!which eole

# 対訳データのインストールと前処理

## 田中コーパス
田中コーパスというデータで翻訳実験を行います。日本の大学生が翻訳したデータです。そのうち適当な長さの5万文だけを抽出して実験しやすいようにしてくれたデータを用います。
train.{ja,en}というのが学習データ、dev.{ja,en}というのが開発データ、test.{ja,en}というのがテストデータです。

In [ ]:
![ -d small_parallel_enja ] || git clone https://github.com/odashi/small_parallel_enja

In [ ]:
!ls small_parallel_enja

## データの前処理
`eole build_vocab` というコマンドでデータの前処理をします。語彙のサイズをコントロールしたり、単語分割をしたりします。YAML という形式で書かれた設定ファイルが必要なので、準備しておきます。以下では tanaka-enja.yaml という名前で設定ファイルを保存します。
Eole の設定ファイルでは、モデルの設定を `model:`、学習の設定を `training:` の下にまとめて書きます。

In [ ]:
%%bash
cat > tanaka-enja.yaml << __EOF__
# 前処理したデータの出力先の設定
save_data: ./tanaka
src_vocab: ./tanaka/vocab.src
tgt_vocab: ./tanaka/vocab.tgt
overwrite: True

# 学習に必要なデータが置いてある置き場所
data:
    corpus_1:
        path_src: small_parallel_enja/train.en
        path_tgt: small_parallel_enja/train.ja
    valid:
        path_src: small_parallel_enja/dev.en
        path_tgt: small_parallel_enja/dev.ja

# モデルの設定（RNN (LSTM) を使う）
model:
    architecture: rnn

# ここから学習時の設定
training:
    world_size: 1
    gpu_ranks: [0]
    model_path: ./tanaka/run/model
    save_checkpoint_steps: 500
    train_steps: 20000
    valid_steps: 500
__EOF__

In [ ]:
!eole build_vocab -c tanaka-enja.yaml -n_sample 10000

# NMTの訓練

## 学習
`eole train` というコマンドで学習します。上述の設定ファイルで挙動を制御できます。いくつかオプションがあるので、変えてみると挙動も変わります。
world_size で使う GPU の個数、gpu_ranks で使う GPU を指定します（0スタートです）。指定しないと CPU になってしまいます。CPU は GPU の数十倍遅いです。
train_steps は学習するステップを指定します。20,000だとこのデータは T4 GPU で30分弱で学習できます（500ステップごとの検証を含みます）。
ここでは `architecture: rnn` として RNN (LSTM) を使って学習します。Transformer を使いたい場合は `architecture: transformer` に変更してみましょう。その他の設定は[設定ファイルの書き方](https://eole-nlp.github.io/eole/docs/concepts/config)や[モデルの設定一覧](https://eole-nlp.github.io/eole/docs/reference/Config/models)を見て編集してみましょう。（ただし、今回はデータが少ないので Transformer のデフォルト設定で学習しても必ずしも精度は高くなりません）

In [ ]:
!eole train -c tanaka-enja.yaml

## 翻訳モデル
学習されたモデルは tanaka/run/model の下に step_10000 のような名前のディレクトリで保存されています。tanaka/run/model 自体を指定すると、最後に保存されたモデルが使われます。

In [ ]:
!ls tanaka/run/model

# NMTによる翻訳

## デコード
`eole predict` というコマンドでデコードします。`--model_path` で一つ前に学習した翻訳モデルのうちの1つを指定します。
`--src` で翻訳したい入力ファイル、`--output pred.txt` で結果を保存するファイルを指定します。
`--gpu_ranks 0` で GPU を使って翻訳します。
`--verbose` は結果を冗長に画面にも表示させるオプションです。

In [ ]:
!eole predict --model_path tanaka/run/model/step_20000 --src small_parallel_enja/test.en --output pred.txt --gpu_ranks 0 --verbose

## BLEU による評価
正解（参照訳）のファイルを第一引数に、システムの出力をリダイレクト（<）で渡します。
sacrebleu というツールで評価します。Eole と一緒に仮想環境にインストールされているので、そのまま使えます。

In [ ]:
!sacrebleu small_parallel_enja/test.ja < pred.txt

## モデル選択

どのモデルがいいかというのは未知のデータで評価する必要があります。ここでは開発データとテストデータが用意されているので、開発データでもっとも性能がよいモデルでテストデータの翻訳をしてみます。2026年10月に試したときは、開発データでの BLEU は20000ステップのモデルが35.4で一番高く、次いで16000ステップのモデルが35.1でした。下の例では20000ステップのモデルを使っています。学習のたびに結果は少し変わるので、開発データでの結果を見て、一番よいステップに変えてみてください。

In [ ]:
!for step in {1..20}000; do echo ${step}; eole predict --model_path tanaka/run/model/step_${step} --src small_parallel_enja/dev.en --output pred_step_${step}.txt --gpu_ranks 0; sacrebleu small_parallel_enja/dev.ja < pred_step_${step}.txt; done

In [ ]:
!eole predict --model_path tanaka/run/model/step_20000 --src small_parallel_enja/test.en --output pred.txt --gpu_ranks 0; sacrebleu small_parallel_enja/test.ja < pred.txt

## 結果の保存

画面に表示して保存する場合は `cat` というコマンドを使って表示させます。複数の結果を表示させたい場合は `paste` というコマンドを使います。
結果をファイルに保存する場合は `files.download` というメソッドを呼びます。
出力されていない結果は24時間で消えるので、ご注意ください。

In [ ]:
!paste pred.txt small_parallel_enja/test.{ja,en}

In [ ]:
from google.colab import files
files.download("pred.txt")